# Modelni qo‘shimcha o‘qitish, dekoderni tanlash va benchmarkda baholash

Fayl: `08_finetune_and_evaluate_benchmark.ipynb`. Ishga tushirish va papkalar izohi: [README.md](README.md).

CUDA GPU va Python kutubxonalari talab qilinadi. Model/dekoder faqat ichki validationda tanlanadi; benchmark yakuniy baholashga ishlatiladi. `RUN_BENCHMARK=False` va `RUN_FINAL_TEST=False` dastlab o‘chiq.


In [1]:
from pathlib import Path
import io, json, math, random, hashlib, unicodedata
from collections import Counter, OrderedDict
from itertools import product
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from peft import LoraConfig, get_peft_model
from pyctcdecode import build_ctcdecoder
from jiwer import wer, cer, process_words
from tqdm.auto import tqdm

PROJECT = Path.cwd().resolve()
assert (PROJECT / "data/train_metadata.parquet").exists(), "Loyiha ildizidan oching."
CHECKPOINT = PROJECT / "checkpoints/full_lora_step_8500"
BASE_MODEL = "facebook/wav2vec2-xls-r-300m"
OUT = PROJECT / "results/benchmark_blind_improvement"
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
BUILD_EXTRA_LM = True  # Lokal KenLM bilan tozalangan train-only 3/5-gram.
DO_FINETUNE = False
MAX_EPOCHS = 6  # Qo‘shimcha fine-tuning uchun maksimal epoch soni.
RUN_FINAL_TEST = False  # Faqat validation tanlovi tugagandan keyin True.

/home/hasan-hafizov/miniconda3/envs/audio/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def normalize_text(value):
    if not isinstance(value, str):
        raise ValueError("Matn string bo‘lishi kerak; NaN ni 'nan'ga aylantirmang.")
    return " ".join(unicodedata.normalize("NFC", value).lower().split())

processor = Wav2Vec2Processor.from_pretrained(
    str(CHECKPOINT), local_files_only=True
)
tokenizer = processor.tokenizer
vocab = tokenizer.get_vocab()
blank_id = tokenizer.pad_token_id
allowed_chars = {t for t in vocab if len(t) == 1 and t != "|"}
assert set(vocab.values()) == set(range(len(vocab)))
assert blank_id is not None and tokenizer.word_delimiter_token == "|"

labels = [tokenizer.convert_ids_to_tokens(i) for i in range(len(vocab))]
labels[blank_id] = ""  # CTC blank aynan PAD ustunida.
labels[vocab["|"]] = " "
assert len(set(labels)) == len(labels), "Decoder label takrorlangan."

In [3]:
frames = {}
for split in ("train", "val", "test"):
    df = pd.read_parquet(PROJECT / f"data/{split}_metadata.parquet")
    required = {"id", "cleaned_text_lowercase", "source_file", "source_row"}
    assert required <= set(df.columns), f"{split}: ustunlar yetishmaydi."
    assert df[list(required - {"cleaned_text_lowercase"})].notna().all().all()
    rows = pd.to_numeric(df["source_row"], errors="raise")
    assert ((rows >= 0) & (rows == np.floor(rows))).all()
    df["source_row"] = rows.astype(int)
    df["text"] = df["cleaned_text_lowercase"].map(
        lambda t: normalize_text(t) if isinstance(t, str) else ""
    )
    df["bad_text"] = df["text"].map(
        lambda t: not t or any(c not in allowed_chars and c != " " for c in t)
    )
    assert not df["id"].duplicated().any(), f"{split}: takror ID."
    if split != "train":
        assert not df["bad_text"].any(), f"{split}: reference matnini tekshiring."
        assert not df.duplicated(["source_file", "source_row"]).any()
    frames[split] = df

for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
    assert not (set(frames[left]["id"]) & set(frames[right]["id"])), "Split ID leakage."
    loc = lambda d: set(zip(d["source_file"], d["source_row"]))
    assert not (loc(frames[left]) & loc(frames[right])), "Split audio leakage."

train_df = frames["train"].copy()
heldout_texts = set(frames["val"]["text"]) | set(frames["test"]["text"])
remove = (train_df["bad_text"] | train_df.duplicated(["source_file", "source_row"])
          | train_df["text"].isin(heldout_texts))
train_df.loc[remove].to_csv(OUT / "train_quarantine.csv", index=False)
train_df = train_df.loc[~remove].reset_index(drop=True)
val_df = frames["val"].reset_index(drop=True)
test_df = frames["test"].reset_index(drop=True)
assert len(train_df) and len(val_df) and len(test_df)
train_df.to_parquet(OUT / "train_clean_metadata.parquet", index=False)

# Audio fayl mavjudligi va source_row chegarasi — audio decoding qilmasdan.
all_meta = pd.concat(list(frames.values()), ignore_index=True)
for name, group in all_meta.groupby("source_file", sort=False):
    shard = PROJECT / "audio_shards" / name
    assert shard.is_file(), f"Shard yo‘q: {shard}"
    count = pq.ParquetFile(shard).metadata.num_rows
    assert group["source_row"].max() < count, f"source_row noto‘g‘ri: {name}"
print({"train": len(train_df), "val": len(val_df), "test": len(test_df),
       "train_quarantine": int(remove.sum())})

{'train': 17362, 'val': 965, 'test': 965, 'train_quarantine': 0}


In [4]:
class AudioDataset(Dataset):
    def __init__(self, metadata, cache_size=4):
        self.df = metadata.reset_index(drop=True)
        self.files, self.ends = {}, {}
        self.cache = OrderedDict()
        self.cache_size = cache_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[int(index)]
        name, position = row["source_file"], int(row["source_row"])
        if name not in self.files:
            self.files[name] = pq.ParquetFile(PROJECT / "audio_shards" / name)
            meta = self.files[name].metadata
            self.ends[name] = np.cumsum([
                meta.row_group(i).num_rows for i in range(meta.num_row_groups)
            ])
        ends = self.ends[name]
        assert 0 <= position < ends[-1]
        group = int(np.searchsorted(ends, position, side="right"))
        local = position - (int(ends[group - 1]) if group else 0)
        key = (name, group)
        if key not in self.cache:
            self.cache[key] = self.files[name].read_row_group(
                group, columns=["id", "audio"]
            ).to_pylist()
            if len(self.cache) > self.cache_size:
                self.cache.popitem(last=False)
        self.cache.move_to_end(key)
        record = self.cache[key][local]
        assert str(record["id"]) == str(row["id"]), "Audio/reference ID mos emas."
        payload = record["audio"]
        if isinstance(payload, dict):
            payload = payload.get("bytes")
        if not isinstance(payload, (bytes, bytearray)):
            raise ValueError(f"Audio bytes yo‘q: {row['id']}")
        audio, sr = sf.read(io.BytesIO(payload), dtype="float32")
        if audio.ndim == 2:
            audio = audio.mean(axis=1)
        if not len(audio) or not np.isfinite(audio).all() or sr <= 0:
            raise ValueError(f"Audio buzilgan: {row['id']}")
        if sr != 16000:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
        if not np.any(audio):
            raise ValueError(f"Audio to‘liq jim: {row['id']}")
        return {"id": row["id"], "audio": np.asarray(audio, dtype=np.float32),
                "text": row["text"]}

train_data = AudioDataset(train_df)
val_data = AudioDataset(val_df)
test_data = AudioDataset(test_df)

In [ ]:
# Qayta bajarilganda shu kerneldagi eski model nusxasini bo‘shatish.
import gc
for old_name in ("model", "base"):
    old_model = globals().pop(old_name, None)
    if old_model is not None:
        old_model.to("cpu")
    del old_model
for old_name in ("optimizer", "scheduler", "loader", "trainable", "parameter",
                 "checkpoint", "weights", "current", "selected_state", "loss", "batch"):
    globals().pop(old_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

checkpoint_path = CHECKPOINT / "training_state.pt"
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
assert int(checkpoint["global_step"]) == 8500
assert checkpoint["vocab_size"] == len(vocab)
assert checkpoint["base_model_name"] == "facebook/wav2vec2-xls-r-300m"
base = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL, local_files_only=True,
    vocab_size=len(vocab), pad_token_id=blank_id,
    ignore_mismatched_sizes=True,
    ctc_loss_reduction="mean", ctc_zero_infinity=False
)
for parameter in base.parameters():
    parameter.requires_grad = False
lora = LoraConfig.from_pretrained(str(CHECKPOINT))
lora.inference_mode = False
model = get_peft_model(base, lora)
weights = checkpoint["model_weights"]
current = model.state_dict()
required_weights = {name for name, p in model.named_parameters() if p.requires_grad}
assert required_weights <= set(weights), "LoRA yoki CTC head og‘irliklari yetishmaydi."
assert any("lm_head" in name for name in required_weights)
for name, value in weights.items():
    assert name in current and current[name].shape == value.shape, name
loaded = model.load_state_dict(weights, strict=False)
assert not loaded.unexpected_keys
assert not (required_weights & set(loaded.missing_keys))
if DEVICE == "cuda":
    required_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    required_bytes += sum(b.numel() * b.element_size() for b in model.buffers())
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"GPU bo‘sh: {free_bytes / 2**30:.2f}/{total_bytes / 2**30:.2f} GiB")
    if free_bytes < required_bytes + 256 * 2**20:
        raise RuntimeError(
            "GPUda modelni yuklash uchun joy yetmaydi. Boshqa notebooklarning GPU "
            "kernellarini Shut Down qiling (tabni yopish yetmaydi). "
            "So‘ng shu kernelni Restart qilib celllarni boshidan bajaring. "
            "Boshqa jarayon avtomatik to‘xtatilmadi.")
try:
    model.to(DEVICE).eval()
except torch.cuda.OutOfMemoryError as exc:
    model.to("cpu")
    gc.collect()
    torch.cuda.empty_cache()
    raise RuntimeError("GPU xotirasi tugadi. Eski GPU kernellarini Shut Down qiling va qayta boshlang.") from exc
model_tag = "step8500"
del checkpoint, weights, current, base

In [ ]:
def collate_ctc(samples):
    batch = processor([s["audio"] for s in samples], sampling_rate=16000,
                      padding=True, return_attention_mask=True, return_tensors="pt")
    targets = tokenizer([s["text"] for s in samples], padding=True,
                        add_special_tokens=False, return_tensors="pt")
    labels_ctc = targets.input_ids.masked_fill(targets.attention_mask == 0, -100)
    assert not ((labels_ctc == blank_id) | (labels_ctc == tokenizer.unk_token_id)).any()
    lengths = model.get_base_model()._get_feat_extract_output_lengths(
        batch.attention_mask.sum(-1)
    )
    for ids, length in zip(labels_ctc, lengths):
        ids = ids[ids != -100]
        minimum = len(ids) + int((ids[1:] == ids[:-1]).sum())
        if minimum > int(length):
            raise ValueError("CTC alignment imkonsiz: audio/transcriptni tekshiring.")
    return {**dict(batch), "labels": labels_ctc}

@torch.inference_mode()
def get_logits(sample):
    model.eval()
    inputs = processor(sample["audio"], sampling_rate=16000,
                       return_attention_mask=True, return_tensors="pt")
    logits = model(**{k: v.to(DEVICE) for k, v in inputs.items()}).logits[0]
    result = logits.float().cpu().numpy()
    assert result.ndim == 2 and result.shape[1] == len(vocab)
    assert np.isfinite(result).all()
    return result

def greedy_validation():
    refs, hyps = [], []
    for i in tqdm(range(len(val_data)), desc="Acoustic validation"):
        sample = val_data[i]
        refs.append(sample["text"])
        hyps.append(normalize_text(processor.decode(get_logits(sample).argmax(-1))))
    return {"wer": wer(refs, hyps) * 100, "cer": cer(refs, hyps) * 100}

In [ ]:
if DO_FINETUNE:
    loader = DataLoader(train_data, batch_size=1, shuffle=True,
                        collate_fn=collate_ctc, num_workers=0)
    trainable = [(name, p) for name, p in model.named_parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW([
        {"params": [p for name, p in trainable if "lm_head" not in name], "lr": 1e-5},
        {"params": [p for name, p in trainable if "lm_head" in name], "lr": 3e-5}
    ], weight_decay=0.01)
    best_metric = greedy_validation()
    best_state = {name: p.detach().cpu().clone() for name, p in trainable}
    improved, history = False, []
    accumulation = 4
    for epoch in range(MAX_EPOCHS):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        for index, batch in enumerate(tqdm(loader, desc=f"Epoch {epoch + 1}")):
            # Oxirgi kichik accumulation guruhi ham to‘g‘ri normalizatsiya qilinadi.
            group_start = (index // accumulation) * accumulation
            divisor = min(accumulation, len(loader) - group_start)
            loss = model(**{k: v.to(DEVICE) for k, v in batch.items()}).loss
            if not torch.isfinite(loss):
                raise RuntimeError("Loss NaN/Inf; audio/transcriptni tekshiring.")
            (loss / divisor).backward()
            if (index + 1) % accumulation == 0 or index + 1 == len(loader):
                torch.nn.utils.clip_grad_norm_([p for _, p in trainable], 1.0,
                                              error_if_nonfinite=True)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
        score = greedy_validation()
        history.append({"epoch": epoch + 1, **score})
        # WER kamayishi va CER yomonlashmasligi birgalikda talab qilinadi.
        if score["wer"] < best_metric["wer"] and score["cer"] <= best_metric["cer"]:
            best_metric, improved = score, True
            best_state = {name: p.detach().cpu().clone() for name, p in trainable}
        else:
            break
    model.load_state_dict(best_state, strict=False)
    model.eval()
    if improved:
        model_tag = "step8500_finetuned"
        model.save_pretrained(OUT / "finetuned_adapter", safe_serialization=True)
        processor.save_pretrained(OUT / "finetuned_adapter/processor")
    pd.DataFrame(history).to_csv(OUT / "acoustic_training_history.csv", index=False)

In [ ]:
DO_ROBUST_FINETUNE = True
ROBUST_EPOCHS = 4
ROBUST_ACCUMULATION = 8
ROBUST_EVAL_STEPS = 250  # optimizer qadamlari
ROBUST_PATIENCE = 3
ROBUST_DIR = OUT / "benchmark_blind_acoustic"
ROBUST_DIR.mkdir(parents=True, exist_ok=True)
assert not (DO_FINETUNE and DO_ROBUST_FINETUNE), "Faqat bitta fine-tuning yo‘lini yoqing."

class AugmentedTrainDataset(Dataset):
    def __init__(self, source, seed=SEED):
        self.source = source
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.source)

    def __getitem__(self, index):
        sample = dict(self.source[index])
        audio = sample["audio"].copy()
        # 25% namuna original holatda: faqat train augment qilinadi.
        if self.rng.random() < 0.75:
            speed = float(self.rng.choice([0.9, 1.0, 1.1]))
            if speed != 1.0:
                # Sample-rate perturbation: tezlik/pitch birga o‘zgaradi.
                candidate = librosa.resample(audio, orig_sr=16000 * speed,
                                             target_sr=16000)
                ids = tokenizer(sample["text"], add_special_tokens=False).input_ids
                minimum = len(ids) + sum(a == b for a, b in zip(ids, ids[1:]))
                available = int(model.get_base_model()._get_feat_extract_output_lengths(
                    torch.tensor([len(candidate)]))[0])
                if available >= minimum:
                    audio = candidate
            if self.rng.random() < 0.5:
                rms = np.sqrt(np.mean(audio.astype(np.float64) ** 2))
                noise = self.rng.standard_normal(len(audio)).astype(np.float32)
                noise_rms = np.sqrt(np.mean(noise.astype(np.float64) ** 2))
                snr = float(self.rng.uniform(15.0, 30.0))
                audio = audio + noise * (rms / max(noise_rms, 1e-12)) * 10 ** (-snr / 20)
        assert np.isfinite(audio).all()
        sample["audio"] = np.asarray(audio, dtype=np.float32)
        return sample

In [ ]:
if DO_ROBUST_FINETUNE:
    if DEVICE != "cuda":
        raise RuntimeError("Bu treningni CUDA GPU kernelida bajaring; CPUda boshlanmadi.")
    from transformers import get_cosine_schedule_with_warmup

    # Muzlatilgan bazani fp32 saqlab, activation xotirasini kamaytiramiz.
    model.get_base_model().gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False})
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))

    trainable = [(name, p) for name, p in model.named_parameters() if p.requires_grad]
    assert trainable and any("lm_head" in name for name, _ in trainable)
    loader = DataLoader(AugmentedTrainDataset(train_data), batch_size=1,
                        shuffle=True, collate_fn=collate_ctc, num_workers=0,
                        generator=torch.Generator().manual_seed(SEED))
    optimizer = torch.optim.AdamW([
        {"params": [p for name, p in trainable if "lm_head" not in name], "lr": 5e-6},
        {"params": [p for name, p in trainable if "lm_head" in name], "lr": 1e-5},
    ], weight_decay=0.01)
    steps_per_epoch = math.ceil(len(loader) / ROBUST_ACCUMULATION)
    total_steps = steps_per_epoch * ROBUST_EPOCHS
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=max(1, int(total_steps * 0.1)),
        num_training_steps=total_steps)
    initial_score = greedy_validation()
    selected_score = dict(initial_score)
    selected_state = {name: p.detach().cpu().clone() for name, p in trainable}
    history = [{"step": 0, "epoch": 0, "selected": True, **initial_score}]
    step, stale, selected_step = 0, 0, 0
    stop = False
    try:
        for epoch in range(ROBUST_EPOCHS):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            for index, batch in enumerate(tqdm(loader, desc=f"Robust epoch {epoch + 1}")):
                group_start = (index // ROBUST_ACCUMULATION) * ROBUST_ACCUMULATION
                divisor = min(ROBUST_ACCUMULATION, len(loader) - group_start)
                with torch.autocast(device_type="cuda", dtype=amp_dtype):
                    loss = model(**{k: v.to(DEVICE) for k, v in batch.items()}).loss
                if not torch.isfinite(loss):
                    raise RuntimeError("CTC loss NaN/Inf: trening to‘xtatildi.")
                scaler.scale(loss / divisor).backward()
                update = ((index + 1) % ROBUST_ACCUMULATION == 0 or index + 1 == len(loader))
                if not update:
                    continue
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for _, p in trainable], 1.0,
                                              error_if_nonfinite=not scaler.is_enabled())
                previous_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                did_step = scaler.get_scale() >= previous_scale
                if did_step:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                if not did_step:
                    continue
                step += 1
                if step % ROBUST_EVAL_STEPS == 0 or index + 1 == len(loader):
                    score = greedy_validation()
                    better = (score["wer"] < selected_score["wer"]
                              and score["cer"] <= selected_score["cer"])
                    if better:
                        selected_score = dict(score)
                        selected_step, stale = step, 0
                        selected_state = {name: p.detach().cpu().clone()
                                          for name, p in trainable}
                        torch.save(selected_state, ROBUST_DIR / "best_trainable_weights.pt")
                    else:
                        stale += 1
                    history.append({"step": step, "epoch": epoch + 1,
                                    "selected": better, **score})
                    pd.DataFrame(history).to_csv(ROBUST_DIR / "history.csv", index=False)
                    print({"step": step, "validation": score, "best": selected_score})
                    model.train()
                    if stale >= ROBUST_PATIENCE:
                        stop = True
                        break
            if stop:
                break
    finally:
        # Xato/interrupt bo‘lsa ham kernelda oxirgi tasodifiy model qolmaydi.
        model.load_state_dict(selected_state, strict=False)
        model.eval()
        model.get_base_model().gradient_checkpointing_disable()
        optimizer.zero_grad(set_to_none=True)
    model_tag = "step8500_robust" if selected_step else "step8500"
    model.save_pretrained(ROBUST_DIR / "adapter", safe_serialization=True)
    processor.save_pretrained(ROBUST_DIR / "processor")
    (ROBUST_DIR / "selection.json").write_text(json.dumps({
        "training_split": "data/train_metadata.parquet (audited train_df)",
        "selection_split": "data/val_metadata.parquet", "benchmark_used": False,
        "seed": SEED, "selected_step": selected_step, "baseline": initial_score,
        "selected": selected_score, "model_tag": model_tag,
        "augmentation": "75% speed 0.9/1/1.1; noise probability 0.5, SNR 15-30 dB",
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    del optimizer, scheduler, loader, selected_state, scaler
    torch.cuda.empty_cache()
    print("Tanlangan model:", model_tag, selected_score)

In [ ]:
corpus_path = OUT / "lm_train_only.txt"
train_sentences = sorted(set(train_df["text"]))
assert not (set(train_sentences) & heldout_texts)
corpus_path.write_text("\n".join(train_sentences) + "\n", encoding="utf-8")
lm_paths = {}
if not BUILD_EXTRA_LM:
    existing_corpus = {normalize_text(line) for line in
                       (PROJECT / "kenlm/karakalpak_train.txt").read_text(encoding="utf-8").splitlines()
                       if line.strip()}
    assert not (existing_corpus & heldout_texts), "Mavjud LMda heldout gap bor; BUILD_EXTRA_LM=True qiling."
    assert existing_corpus <= set(train_df["text"]), "Mavjud LM clean train bilan mos emas; qayta quring."
    lm_paths["5gram_existing"] = PROJECT / "kenlm/karakalpak_5gram.binary"
    assert lm_paths["5gram_existing"].is_file()

if BUILD_EXTRA_LM:
    import subprocess
    tool_dir = PROJECT / "kenlm/kenlm_source/build/bin"
    assert (tool_dir / "lmplz").is_file() and (tool_dir / "build_binary").is_file()
    for order in (3, 5):
        arpa = OUT / f"train_only_{order}gram.arpa"
        binary = OUT / f"train_only_{order}gram.binary"
        with corpus_path.open("rb") as source, arpa.open("wb") as target:
            subprocess.run([str(tool_dir / "lmplz"), "-o", str(order),
                            "--discount_fallback", "-S", "25%", "-T", str(OUT)],
                           stdin=source, stdout=target, check=True)
        subprocess.run([str(tool_dir / "build_binary"), str(arpa), str(binary)], check=True)
        lm_paths[f"{order}gram_train_only"] = binary
DEFAULT_LM = "5gram_train_only" if BUILD_EXTRA_LM else "5gram_existing"


In [ ]:
def model_fingerprint():
    digest = hashlib.sha256()
    digest.update(str(BASE_MODEL).encode())
    digest.update(json.dumps(model.config.to_dict(), sort_keys=True, default=str).encode())
    digest.update(json.dumps(model.peft_config["default"].to_dict(),
                             sort_keys=True, default=str).encode())
    digest.update(json.dumps(processor.feature_extractor.to_dict(),
                             sort_keys=True, default=str).encode())
    digest.update(json.dumps(vocab, sort_keys=True, ensure_ascii=False).encode())
    for name, parameter in sorted(model.named_parameters()):
        if parameter.requires_grad:
            digest.update(name.encode())
            digest.update(parameter.detach().cpu().float().contiguous().numpy().tobytes())
    return digest.hexdigest()

model_hash = model_fingerprint()

def make_cache(dataset, split):
    # Source metadata va shard size/mtime o‘zgarishi cache kalitini yangilaydi.
    digest = hashlib.sha256()
    digest.update(dataset.df[["id", "source_file", "source_row", "text"]].to_json().encode())
    for name in sorted(set(dataset.df["source_file"])):
        stat = (PROJECT / "audio_shards" / name).stat()
        digest.update(f"{name}:{stat.st_size}:{stat.st_mtime_ns}".encode())
    folder = OUT / "logits" / model_hash[:16] / f"{split}_{digest.hexdigest()[:16]}"
    folder.mkdir(parents=True, exist_ok=True)
    records = []
    for index in tqdm(range(len(dataset)), desc=f"{split} logits"):
        row = dataset.df.iloc[index]
        path = folder / f"sample_{index}.npy"
        if not path.exists():
            sample = dataset[index]
            np.save(path, get_logits(sample), allow_pickle=False)
        logits = np.load(path, allow_pickle=False)
        assert logits.ndim == 2 and logits.shape[1] == len(vocab)
        assert len(logits) and np.isfinite(logits).all()
        records.append({"id": row["id"], "reference": row["text"],
                        "logits_path": str(path.relative_to(PROJECT))})
    result = pd.DataFrame(records)
    result.to_csv(folder / "manifest.csv", index=False)
    return result

val_cache = make_cache(val_data, "val")
# Validation kichik; tuningda modelni qayta-qayta chaqirmaymiz.
val_logits = [np.load(PROJECT / path, allow_pickle=False)
              for path in val_cache["logits_path"]]
val_refs = val_cache["reference"].tolist()

In [ ]:
def metrics(refs, hypotheses):
    refs = [normalize_text(x) for x in refs]
    hypotheses = [normalize_text(x) for x in hypotheses]
    assert len(refs) == len(hypotheses) and len(refs)
    words = process_words(refs, hypotheses)
    return {"wer": wer(refs, hypotheses) * 100, "cer": cer(refs, hypotheses) * 100,
            "substitutions": words.substitutions, "deletions": words.deletions,
            "insertions": words.insertions, "samples": len(refs)}

decoders = {name: build_ctcdecoder(labels, kenlm_model_path=str(path))
            for name, path in lm_paths.items()}
plain_decoder = build_ctcdecoder(labels)
BASE_CONFIG = {"mode": "kenlm", "lm": DEFAULT_LM, "alpha": 0.65,
               "beta": 1.5, "beam_width": 200, "temperature": 1.0,
               "blank_bias": 0.0, "unk_score_offset": -10.0,
               "beam_prune_logp": -10.0, "token_min_logp": -5.0}

def decode_all(logits_list, config):
    if config["mode"] == "greedy":
        return [normalize_text(processor.decode(x.argmax(-1))) for x in logits_list]
    decoder = plain_decoder if config["mode"] == "beam" else decoders[config["lm"]]
    if config["mode"] == "kenlm":
        decoder.reset_params(alpha=config["alpha"], beta=config["beta"],
                             unk_score_offset=config["unk_score_offset"],
                             lm_score_boundary=True)
    predictions = []
    for raw in logits_list:
        logits = raw.copy() / config["temperature"]
        logits[:, blank_id] += config["blank_bias"]
        text = decoder.decode(logits, beam_width=int(config["beam_width"]),
                              beam_prune_logp=config["beam_prune_logp"],
                              token_min_logp=config["token_min_logp"])
        predictions.append(normalize_text(text))
    return predictions

baseline_predictions = decode_all(val_logits, BASE_CONFIG)
baseline_metrics = metrics(val_refs, baseline_predictions)
comparisons = []
for label, config in [("Greedy", {**BASE_CONFIG, "mode": "greedy"}),
                      ("Beam without LM", {**BASE_CONFIG, "mode": "beam"}),
                      ("Beam + existing KenLM", BASE_CONFIG)]:
    comparisons.append({"decoder": label, **metrics(val_refs, decode_all(val_logits, config))})
display(pd.DataFrame(comparisons))

In [ ]:
trials = []

def evaluate_config(config, stage):
    score = metrics(val_refs, decode_all(val_logits, config))
    trials.append({**config, **score, "stage": stage})
    pd.DataFrame(trials).to_csv(OUT / "validation_tuning.csv", index=False)
    return score

def choose_best():
    valid = [t for t in trials if t["wer"] <= baseline_metrics["wer"]
             and t["cer"] <= baseline_metrics["cer"]]
    winner = min(valid, key=lambda t: (t["wer"], t["cer"], t["beam_width"]))
    return {key: winner[key] for key in BASE_CONFIG}

# Baseline konfiguratsiya ham nomzodlar ichida.
trials.append({**BASE_CONFIG, **baseline_metrics, "stage": "baseline"})
for lm_name, alpha, beta in tqdm(list(product(
        lm_paths, [0.35, 0.5, 0.65, 0.8], [0.5, 1.0, 1.5])), desc="Alpha/beta"):
    evaluate_config({**BASE_CONFIG, "lm": lm_name, "alpha": alpha, "beta": beta}, "lm")
best_config = choose_best()

stage_base = best_config.copy()
for temperature, blank_bias in tqdm(list(product(
        [0.85, 1.0, 1.15], [-0.25, 0.0, 0.25])), desc="Acoustic calibration"):
    evaluate_config({**stage_base, "temperature": temperature,
                     "blank_bias": blank_bias}, "calibration")
best_config = choose_best()

stage_base = best_config.copy()
for beam_width, prune, token_min in [(100, -10.0, -5.0), (300, -10.0, -5.0),
                                    (200, -15.0, -7.0)]:
    evaluate_config({**stage_base, "beam_width": beam_width,
                     "beam_prune_logp": prune, "token_min_logp": token_min}, "beam")
best_config = choose_best()

stage_base = best_config.copy()
for offset in [-5.0, -15.0]:
    evaluate_config({**stage_base, "unk_score_offset": offset}, "unknown_word")
best_config = choose_best()
display(pd.DataFrame(trials).sort_values(["wer", "cer"]).head(10))

In [ ]:
best_val_predictions = decode_all(val_logits, best_config)
best_val_metrics = metrics(val_refs, best_val_predictions)
config_record = {
    "checkpoint": str(CHECKPOINT.relative_to(PROJECT)), "model_tag": model_tag,
    "model_fingerprint": model_hash,
    "lm_path": str(lm_paths[best_config["lm"]].relative_to(PROJECT)),
    "selection_split": "val", "normalization": "NFC + lowercase + whitespace",
    "baseline_validation": baseline_metrics, "selected_validation": best_val_metrics,
    "decoder": best_config
}
(OUT / "selected_decoder.json").write_text(
    json.dumps(config_record, ensure_ascii=False, indent=2), encoding="utf-8"
)
errors = val_cache[["id", "reference"]].copy()
errors["baseline_prediction"] = baseline_predictions
errors["optimized_prediction"] = best_val_predictions
errors["word_errors"] = [
    sum([r.substitutions, r.deletions, r.insertions])
    for r in (process_words(ref, hyp) for ref, hyp in zip(val_refs, best_val_predictions))
]
errors.to_csv(OUT / "validation_predictions.csv", index=False)
display(errors.sort_values("word_errors", ascending=False).head(10))
print("Baseline:", baseline_metrics)
print("Selected:", best_val_metrics)

In [ ]:
# Shu cellni decoder tanlangandan keyin bajaring. Benchmark uchun mustaqil bundle.
import shutil
BUNDLE = OUT / "benchmark_blind_bundle"
BUNDLE.mkdir(parents=True, exist_ok=True)
assert model_fingerprint() == model_hash, "Model o‘zgargan: logits/tuningni qayta bajaring."
model.save_pretrained(BUNDLE / "adapter", safe_serialization=True)
processor.save_pretrained(BUNDLE / "processor")
shutil.copy2(lm_paths[best_config["lm"]], BUNDLE / "kenlm.binary")
bundle_record = {**config_record, "base_model": BASE_MODEL,
                 "adapter_path": "adapter", "processor_path": "processor",
                 "lm_path": "kenlm.binary", "benchmark_used_for_selection": False}
def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()
bundle_record["files_sha256"] = {
    str(p.relative_to(BUNDLE)): file_sha256(p)
    for p in sorted(BUNDLE.rglob("*")) if p.is_file() and p.name != "bundle.json"
}
(BUNDLE / "bundle.json").write_text(
    json.dumps(bundle_record, ensure_ascii=False, indent=2), encoding="utf-8")
print("Benchmarkga tayyor bundle:", BUNDLE)

In [ ]:
if RUN_FINAL_TEST:
    test_cache = make_cache(test_data, "test")
    test_logits = [np.load(PROJECT / path, allow_pickle=False)
                   for path in test_cache["logits_path"]]
    test_refs = test_cache["reference"].tolist()
    raw_refs = test_df["cleaned_text_lowercase"].tolist()
    predictions = test_cache[["id", "reference"]].copy()
    predictions["reference_raw"] = raw_refs
    test_scores = []
    configs = {"greedy": {**BASE_CONFIG, "mode": "greedy"},
               "beam_no_lm": {**BASE_CONFIG, "mode": "beam"},
               "baseline_kenlm": BASE_CONFIG, "optimized_kenlm": best_config}
    for name, config in configs.items():
        hypotheses = decode_all(test_logits, config)
        predictions[name] = hypotheses
        test_scores.append({"decoder": name, **metrics(test_refs, hypotheses),
                            "raw_reference_wer": wer(raw_refs, hypotheses) * 100,
                            "raw_reference_cer": cer(raw_refs, hypotheses) * 100})
    predictions.to_csv(OUT / "test_predictions.csv", index=False)
    score_table = pd.DataFrame(test_scores)
    score_table.to_csv(OUT / "test_metrics.csv", index=False)
    display(score_table)
    base_score = score_table.loc[score_table.decoder == "baseline_kenlm"].iloc[0]
    new_score = score_table.loc[score_table.decoder == "optimized_kenlm"].iloc[0]
    print("WER kamayishi (foiz punkt):", base_score.wer - new_score.wer)
    print("CER kamayishi (foiz punkt):", base_score.cer - new_score.cer)
else:
    print("Test bajarilmadi: RUN_FINAL_TEST=False.")

In [ ]:
RUN_BENCHMARK = False
from pathlib import Path
PROJECT = Path.cwd().resolve()
# Hozirgi tashqi benchmarkning to‘liq manifesti.
# Benchmark faqat RUN_BENCHMARK=True bo‘lganda bajariladi.
BENCHMARK_CSV = (PROJECT / "external_validation/Benchmark/karakalpak_hf_dataset"
                 / "benchmark_all_samples.csv")
benchmark_checked = None
benchmark_manifest_hash = None
BUNDLE = PROJECT / "results/benchmark_blind_improvement/benchmark_blind_bundle"
BENCHMARK_OUT = PROJECT / "results/benchmark_blind_evaluation"

In [ ]:
if RUN_BENCHMARK:
    import hashlib, json, unicodedata
    import numpy as np
    import pandas as pd
    import soundfile as sf
    from tqdm.auto import tqdm

    def benchmark_normalize(value):
        if not isinstance(value, str):
            raise ValueError("Reference string bo‘lishi kerak.")
        return " ".join(unicodedata.normalize("NFC", value).lower().split())

    def resolve_benchmark_audio(value):
        if not isinstance(value, str) or not value.strip():
            raise ValueError("audio_path bo‘sh yoki string emas.")
        path = Path(value.strip())
        candidates = [path] if path.is_absolute() else [BENCHMARK_CSV.parent / path,
                                                       PROJECT / path]
        # Boshqa serverdan ko‘chirilgan absolyut loyiha yo‘li.
        if path.is_absolute() and "external_validation" in path.parts:
            part = path.parts.index("external_validation")
            candidates.append(PROJECT.joinpath(*path.parts[part:]))
        existing = {p.resolve() for p in candidates if p.is_file()}
        if len(existing) != 1:
            raise ValueError(f"Audio yo‘li topilmadi yoki noaniq: {value}")
        return str(existing.pop())

    BENCHMARK_OUT.mkdir(parents=True, exist_ok=True)
    if BENCHMARK_CSV is None:
        candidates = sorted((PROJECT / "external_validation").glob("*.csv"))
        if len(candidates) != 1:
            raise ValueError("BENCHMARK_CSVga yangi Benchmark CSVning aniq yo‘lini yozing.")
        BENCHMARK_CSV = candidates[0]
    BENCHMARK_CSV = Path(BENCHMARK_CSV).resolve()
    benchmark_manifest_hash = hashlib.sha256(BENCHMARK_CSV.read_bytes()).hexdigest()
    benchmark_checked = None
    benchmark_df = pd.read_csv(BENCHMARK_CSV, encoding="utf-8-sig", keep_default_na=False)
    assert {"audio_path", "text"} <= set(benchmark_df.columns)
    assert len(benchmark_df), "Benchmark bo‘sh."
    audited, issues = [], []
    seen_paths, seen_audio = set(), set()
    for row_number, row in tqdm(benchmark_df.iterrows(), total=len(benchmark_df), desc="Benchmark audit"):
        try:
            reference = benchmark_normalize(row["text"])
            if not reference:
                raise ValueError("Bo‘sh reference.")
            path = resolve_benchmark_audio(row["audio_path"])
            audio, sr = sf.read(path, dtype="float32", always_2d=True)
            if sr <= 0 or not len(audio) or not np.isfinite(audio).all() or not np.any(audio):
                raise ValueError("Audio bo‘sh, jim yoki NaN/Inf.")
            mono = np.ascontiguousarray(audio.mean(axis=1), dtype=np.float32)
            if not np.any(mono):
                raise ValueError("Mono audio to‘liq jim.")
            audio_hash = hashlib.sha256(str(sr).encode() + mono.tobytes()).hexdigest()
            if path in seen_paths or audio_hash in seen_audio:
                raise ValueError("Takror audio: benchmark tarkibini qo‘lda tasdiqlang.")
            seen_paths.add(path)
            seen_audio.add(audio_hash)
            audited.append({"row": row_number, "audio_path_original": row["audio_path"],
                            "audio_path": path, "reference_raw": row["text"],
                            "reference": reference, "duration_original": row.get("duration", ""),
                            "duration": len(audio) / sr, "sample_rate": sr,
                            "channels": audio.shape[1], "audio_sha256": audio_hash})
        except Exception as exc:
            issues.append({"row": row_number, "audio_path": row["audio_path"], "error": str(exc)})
    pd.DataFrame(issues, columns=["row", "audio_path", "error"]).to_csv(
        BENCHMARK_OUT / "audit_issues.csv", index=False)
    if issues:
        raise ValueError(f"{len(issues)} texnik xato. audit_issues.csvni tekshiring; qatorlar tashlanmadi.")
    benchmark_checked = pd.DataFrame(audited)
    assert len(benchmark_checked) == len(benchmark_df)
    benchmark_checked.to_csv(BENCHMARK_OUT / "benchmark_checked.csv", index=False)
    print("Audit tayyor:", len(benchmark_checked), "namuna. Original CSV saqlandi.")

In [ ]:
if RUN_BENCHMARK:
    import torch, librosa
    from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
    from peft import PeftModel
    from pyctcdecode import build_ctcdecoder
    from jiwer import wer, cer, process_words

    assert benchmark_checked is not None, "Avval yangi audit cellini bajaring."
    assert hashlib.sha256(BENCHMARK_CSV.read_bytes()).hexdigest() == benchmark_manifest_hash, "CSV o‘zgardi: auditni qayta bajaring."
    bundle = json.loads((BUNDLE / "bundle.json").read_text())
    assert bundle["selection_split"] == "val"
    assert bundle["benchmark_used_for_selection"] is False
    for filename, expected in bundle["files_sha256"].items():
        digest = hashlib.sha256()
        with (BUNDLE / filename).open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)
        assert digest.hexdigest() == expected, f"Bundle o‘zgargan: {filename}"
    benchmark_processor = Wav2Vec2Processor.from_pretrained(
        str(BUNDLE / bundle["processor_path"]), local_files_only=True)
    benchmark_tokenizer = benchmark_processor.tokenizer
    benchmark_base = Wav2Vec2ForCTC.from_pretrained(
        bundle["base_model"], local_files_only=True,
        vocab_size=len(benchmark_tokenizer), pad_token_id=benchmark_tokenizer.pad_token_id,
        ignore_mismatched_sizes=True)
    benchmark_model = PeftModel.from_pretrained(
        benchmark_base, str(BUNDLE / bundle["adapter_path"]), local_files_only=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    benchmark_model.to(device).eval()
    labels = [benchmark_tokenizer.convert_ids_to_tokens(i) for i in range(len(benchmark_tokenizer))]
    labels[benchmark_tokenizer.pad_token_id] = ""
    labels[benchmark_tokenizer.word_delimiter_token_id] = " "
    config = bundle["decoder"]
    assert config["mode"] == "kenlm"
    decoder = build_ctcdecoder(labels, kenlm_model_path=str(BUNDLE / bundle["lm_path"]))
    decoder.reset_params(alpha=config["alpha"], beta=config["beta"],
                         unk_score_offset=config["unk_score_offset"], lm_score_boundary=True)
    results = []
    with torch.inference_mode():
        for row in tqdm(benchmark_checked.to_dict("records"), desc="Frozen Benchmark evaluation"):
            audio, sr = sf.read(row["audio_path"], dtype="float32", always_2d=True)
            audio = audio.mean(axis=1)
            if sr != 16000:
                audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
            inputs = benchmark_processor(audio, sampling_rate=16000,
                                         return_attention_mask=True, return_tensors="pt")
            raw = benchmark_model(**{k: v.to(device) for k, v in inputs.items()}).logits[0].float().cpu().numpy()
            assert len(raw) and np.isfinite(raw).all()
            greedy = benchmark_normalize(benchmark_processor.decode(raw.argmax(-1)))
            calibrated = raw / config["temperature"]
            calibrated[:, benchmark_tokenizer.pad_token_id] += config["blank_bias"]
            prediction = decoder.decode(calibrated, beam_width=int(config["beam_width"]),
                                        beam_prune_logp=config["beam_prune_logp"],
                                        token_min_logp=config["token_min_logp"])
            results.append({**row, "greedy_prediction": greedy,
                            "kenlm_prediction": benchmark_normalize(prediction)})
    predictions = pd.DataFrame(results)
    assert len(predictions) == len(benchmark_checked)
    scores = []
    for method in ("greedy", "kenlm"):
        refs = predictions["reference"].tolist()
        hyps = predictions[f"{method}_prediction"].tolist()
        counts = process_words(refs, hyps)
        scores.append({"method": method, "samples": len(refs),
                       "wer": wer(refs, hyps) * 100, "cer": cer(refs, hyps) * 100,
                       "substitutions": counts.substitutions, "deletions": counts.deletions,
                       "insertions": counts.insertions})
    predictions.to_csv(BENCHMARK_OUT / "predictions.csv", index=False)
    pd.DataFrame(scores).to_csv(BENCHMARK_OUT / "metrics.csv", index=False)
    (BENCHMARK_OUT / "evaluation_provenance.json").write_text(json.dumps({
        "bundle": bundle, "benchmark_csv": str(BENCHMARK_CSV),
        "benchmark_csv_sha256": benchmark_manifest_hash,
        "normalization": "NFC + lowercase + whitespace", "samples": len(predictions),
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    display(pd.DataFrame(scores))